# Customer Lifetime Value — EDA & Modelling

**Dataset:** UCI Online Retail II  
**Goal:** Predict 12-month revenue potential per customer using RFM features + XGBoost  
**Observation window:** Jan 2010 – Dec 2010  
**Prediction window:** Jan 2011 – Mar 2011

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebook
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap
from scipy.stats import pearsonr
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from preprocess import (
    load_data, compute_return_rates, clean_data, engineer_features,
    FEATURE_COLS, OBSERVATION_START, OBSERVATION_END,
    PREDICTION_START, PREDICTION_END, RAW_FILE, ROOT_DIR
)
from train import evaluate_metrics, mape, build_pipeline, generate_shap_plots

OUTPUTS_DIR = ROOT_DIR / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('Setup complete.')

---
## Step 1: Data Loading & Cleaning

In [ ]:
df_raw = load_data(RAW_FILE)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
print('dtypes:')
print(df_raw.dtypes)
print(f'\nNull counts:\n{df_raw.isnull().sum()}')

In [ ]:
# Compute return rates BEFORE cleaning (includes cancellations)
return_rates = compute_return_rates(df_raw)
print(f'Return rate stats:\n{return_rates.describe()}')

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

df_clean = clean_data(df_raw)
print(f'\nSample after cleaning:')
df_clean.head(3)

---
## Step 2: Feature Engineering

In [ ]:
features = engineer_features(df_clean, return_rates)
print(f'Feature matrix shape: {features.shape}')
features.head()

In [ ]:
print('Feature summary statistics:')
features[FEATURE_COLS + ['future_revenue']].describe().round(2)

In [ ]:
print(f"Customers with future revenue > 0: {(features['future_revenue'] > 0).sum():,}")
print(f"High-value customers (top 25%):    {features['high_value_customer'].sum():,}")
print(f"High-value threshold: £{features['future_revenue'].quantile(0.75):,.2f}")

---
## Step 3: Exploratory Data Analysis

### Plot 1 — Distribution of CLV (future_revenue) on a log scale

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
nonzero = features.loc[features['future_revenue'] > 0, 'future_revenue']
ax.hist(np.log1p(nonzero), bins=50, color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xlabel('log(1 + Future Revenue) [£]')
ax.set_ylabel('Number of Customers')
ax.set_title('Distribution of CLV (future_revenue) — log scale')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'eda_clv_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved eda_clv_distribution.png')

### Plot 2 — RFM Correlation Heatmap

In [ ]:
rfm_cols = ['recency', 'frequency', 'monetary_mean', 'monetary_total',
            'avg_days_between_orders', 'num_unique_products', 'future_revenue']
corr = features[rfm_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, ax=ax
)
ax.set_title('RFM Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'eda_rfm_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved eda_rfm_correlation.png')

### Plot 3 — Top 10 Countries by Revenue

In [ ]:
country_revenue = (
    df_clean
    .groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(country_revenue.index[::-1], country_revenue.values[::-1],
               color='steelblue')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
ax.set_xlabel('Total Revenue')
ax.set_title('Top 10 Countries by Revenue')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'eda_top_countries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved eda_top_countries.png')

### Plot 4 — Monthly Revenue Trend

In [ ]:
monthly = (
    df_clean
    .set_index('InvoiceDate')['Revenue']
    .resample('ME')
    .sum()
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly.index, monthly.values, marker='o', linewidth=1.5, markersize=4,
        color='steelblue')
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='steelblue')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}K'))
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
ax.set_title('Monthly Revenue Trend')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'eda_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved eda_monthly_revenue.png')

### Plot 5 — Frequency vs. Monetary (coloured by CLV quartile)

In [ ]:
plot_df = features[features['future_revenue'] > 0].copy()
plot_df['clv_quartile'] = pd.qcut(
    plot_df['future_revenue'], q=4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']
)

palette = {'Q1 (Low)': '#e74c3c', 'Q2': '#e67e22', 'Q3': '#3498db', 'Q4 (High)': '#2ecc71'}

fig, ax = plt.subplots(figsize=(8, 5))
for q, grp in plot_df.groupby('clv_quartile', observed=True):
    ax.scatter(
        grp['frequency'], grp['monetary_total'],
        label=str(q), alpha=0.5, s=15, color=palette[str(q)]
    )
ax.set_xlabel('Frequency (# invoices)')
ax.set_ylabel('Monetary Total (£)')
ax.set_title('Frequency vs. Monetary — coloured by CLV Quartile')
ax.legend(title='CLV Quartile')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'eda_freq_vs_monetary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved eda_freq_vs_monetary.png')

---
## Step 4: Modelling

In [ ]:
RANDOM_STATE = 42
TARGET_COL = 'future_revenue'

X = features[FEATURE_COLS]
y = features[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

In [ ]:
pipeline = build_pipeline()

# 5-fold CV RMSE
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rmse = np.sqrt(
    -cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')
)
print(f'5-fold CV RMSE: {cv_rmse.mean():.2f} +/- {cv_rmse.std():.2f}')

pipeline.fit(X_train, y_train)
print('Model fitted.')

In [ ]:
y_pred = pipeline.predict(X_test)
metrics = evaluate_metrics(y_test.values, y_pred)

print('\n=== Test Set Metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Actual vs. Predicted
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.4, s=12, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
ax.set_xlabel('Actual Revenue (£)')
ax.set_ylabel('Predicted Revenue (£)')
ax.set_title('Actual vs. Predicted CLV')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

# Save model and artefacts
joblib.dump(pipeline, OUTPUTS_DIR / 'model.pkl')
features.to_parquet(OUTPUTS_DIR / 'features.parquet')
pd.DataFrame({'y_test': y_test.values, 'y_pred': y_pred}).to_parquet(
    OUTPUTS_DIR / 'test_predictions.parquet', index=False
)
with open(OUTPUTS_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Model and artefacts saved to outputs/')

---
## Step 5: SHAP Explainability

In [ ]:
generate_shap_plots(pipeline, X)

In [ ]:
# Inline beeswarm for the notebook
xgb_model = pipeline.named_steps['model']
scaler    = pipeline.named_steps['scaler']
X_scaled  = pd.DataFrame(scaler.transform(X), columns=FEATURE_COLS, index=X.index)

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_scaled)

shap.plots.beeswarm(shap_values, max_display=10)

---
## Summary

| Metric | Value |
|--------|-------|
| MAE | see metrics.json |
| RMSE | see metrics.json |
| R² | see metrics.json |
| MAPE (%) | see metrics.json |
| Pearson r | see metrics.json |

All plots and the trained model are saved to `outputs/`.  
Run `streamlit run app/streamlit_app.py` to explore results interactively.